In [1]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../")

import pandas as pd
from loguru import logger
from imblearn.combine import SMOTETomek

import src.preprocessing.functions as preprocessing_functions
from src.global_vars import BASE_DATA_DIR

data_root_dir = f"{BASE_DATA_DIR}/sun_et_al_data/"
columns_to_keep = ["Sample", "Group", "Project", "Project_1"]
studies_to_remove = ["LiS_2021a", "LiS_2021b"]


def print_full_df(x):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    pd.set_option("display.max_colwidth", None)
    display(x)
    pd.reset_option("display.max_rows")
    pd.reset_option("display.max_columns")
    pd.reset_option("display.width")
    pd.reset_option("display.float_format")
    pd.reset_option("display.max_colwidth")


import os
import sys
from importlib import import_module

import pandas as pd
import torch
from sklearn.preprocessing import Normalizer
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, RandomSampler

# Local imports
sys.path.append("../../")
import src.data.sun_et_al as hf


c:\Users\shaya\Documents\TU_projects\master_thesis\.thesis_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import sys
import pandas as pd
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, RandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, accuracy_score


# Local imports
import src.data.sun_et_al as hf


def train_on_single_study(
    abundance_file: str,
    metadata_file: str,
    study_name: str,
    data_root_dir: str,
    n_epochs: int = 5,
    batch_size: int = 32,
    lr: float = 1e-3,
    model_script: str = "src.models.my_models",  # Adjust to your actual model script
    model_name: str = "SimpleNet",               # Adjust to the model you want
    test_fraction: float = 0.2,
    scale_factor_before_training: float = 100.0,
):
    """
    Train and test on a single study (specified by 'study_name'),
    ignoring all other studies in the dataset.

    This script:
        1. Loads the abundance and metadata CSVs.
        2. Filters them so only samples from 'study_name' remain.
        3. Splits those samples into train and test sets (by 'test_fraction').
        4. Trains a simple neural net on the train portion.
        5. Evaluates on the test portion.

    Args:
        abundance_file (str): Filename of the abundance CSV, containing samples x features.
        metadata_file (str): Filename of the metadata CSV, containing 'Project_1' column with study IDs.
        study_name (str): The name of the single study to train/test on (must appear in 'Project_1').
        data_root_dir (str): Directory path for the data files.
        n_epochs (int): Number of training epochs.
        batch_size (int): Mini-batch size for training.
        lr (float): Learning rate for Adam.
        model_script (str): Python module path for the model definition (has a function get_model(model_name)).
        model_name (str): The model architecture name to instantiate from model_script.
        test_fraction (float): Fraction of data to be used as test split (e.g. 0.2 = 20% test).
        scale_factor_before_training (float): Multiply input features by this factor prior to training.
    """

    # 1) Load the data
    abundance_df = pd.read_csv(os.path.join(data_root_dir, abundance_file), index_col=0)
    metadata_df  = pd.read_csv(os.path.join(data_root_dir, metadata_file), index_col=0)

    # Ensure data sorted by index
    abundance_df.sort_index(inplace=True)
    metadata_df.sort_index(inplace=True)

    # 2) Filter metadata to only the desired study, then subset abundance
    metadata_df = metadata_df[metadata_df["Project_1"] == study_name]
    abundance_df = abundance_df.loc[metadata_df.index]

    # 3) Train/test split within this single study
    #    We'll do a simple random split; any metadata column can be used for stratification, 
    #    but here we just do a naive random split.
    train_ids, test_ids = train_test_split(
        metadata_df.index, test_size=test_fraction, random_state=42, shuffle=True
    )
    train_df = abundance_df.loc[train_ids]
    test_df  = abundance_df.loc[test_ids]
    train_meta = metadata_df.loc[train_ids]
    test_meta  = metadata_df.loc[test_ids]

    # Optionally normalize
    train_df = pd.DataFrame(Normalizer().fit_transform(train_df), index=train_df.index, columns=train_df.columns)
    test_df  = pd.DataFrame(Normalizer().fit_transform(test_df), index=test_df.index, columns=test_df.columns)

    # Optionally scale
    train_df *= scale_factor_before_training
    test_df  *= scale_factor_before_training

    # Rename columns to fit MicrobiomeDataset requirement: "label" and "project"
    train_meta = train_meta.rename(columns={"Group": "label", "Project_1": "project"})
    test_meta  = test_meta.rename(columns={"Group": "label", "Project_1": "project"})

    # 4) Create datasets/dataloaders
    train_ds = hf.MicrobiomeDataset(train_df, train_meta)
    test_ds  = hf.MicrobiomeDataset(test_df, test_meta)

    train_loader = DataLoader(
        train_ds, sampler=RandomSampler(train_ds), batch_size=batch_size
    )
    test_loader = DataLoader(
        test_ds, sampler=RandomSampler(test_ds), batch_size=batch_size
    )

    # Load the model
    from importlib import import_module
    model_module = import_module(model_script)

    n_features = train_df.shape[1]
    model = model_module.get_model(model_name)(n_features)

    # Setup training
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = Adam(model.parameters(), lr=lr)
    loss_fn   = nn.BCEWithLogitsLoss()  # for binary classification

    # 5) Evaluate on test set
    model.eval()
    y_true_test = []
    y_prob_test = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb).squeeze()
            probs = torch.sigmoid(logits)
            y_true_test.extend(yb.cpu().numpy().tolist())
            y_prob_test.extend(probs.cpu().numpy().tolist())


    y_pred_test = [1 if p > 0.5 else 0 for p in y_prob_test]
    accuracy_te = accuracy_score(y_true_test, y_pred_test)
    f1_te = f1_score(y_true_test, y_pred_test)
    recall_te = recall_score(y_true_test, y_pred_test)
    precision_te = precision_score(y_true_test, y_pred_test)
    roc_auc_te = roc_auc_score(y_true_test, y_prob_test)
    print(f"Test before training - Accuracy: {accuracy_te:.3f}, F1: {f1_te:.3f}, Recall: {recall_te:.3f}, Precision: {precision_te:.3f}, ROC-AUC: {roc_auc_te:.3f}")

    y_true_train = []
    y_prob_train = []
    for epoch in range(n_epochs):
        total_loss = 0.0
        model.train()
        for X, y in train_loader:
            X, y = X.to(device, dtype=torch.float), y.to(device, dtype=torch.float)
            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits.squeeze(), y.float())  # shape handling
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(train_loader))
        print(f"[Epoch {epoch+1}/{n_epochs}] Training loss: {avg_loss:.4f}")

        model.eval()
        with torch.no_grad():
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                logits = model(Xb).squeeze()
                probs = torch.sigmoid(logits)  # probabilities for ROC AUC
                y_true_train.extend(yb.cpu().numpy().tolist())
                y_prob_train.extend(probs.cpu().numpy().tolist())

    y_pred_train = [1 if p > 0.5 else 0 for p in y_prob_train]
    accuracy_tr = accuracy_score(y_true_train, y_pred_train)
    f1_tr = f1_score(y_true_train, y_pred_train)
    recall_tr = recall_score(y_true_train, y_pred_train)
    precision_tr = precision_score(y_true_train, y_pred_train)
    roc_auc_tr = roc_auc_score(y_true_train, y_prob_train)
    print(f"Train - Accuracy: {accuracy_tr:.3f}, F1: {f1_tr:.3f}, Recall: {recall_tr:.3f}, Precision: {precision_tr:.3f}, ROC-AUC: {roc_auc_tr:.3f}")

    # 5) Evaluate on test set
    model.eval()
    y_true_test = []
    y_prob_test = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb).squeeze()
            probs = torch.sigmoid(logits)
            y_true_test.extend(yb.cpu().numpy().tolist())
            y_prob_test.extend(probs.cpu().numpy().tolist())


    y_pred_test = [1 if p > 0.5 else 0 for p in y_prob_test]
    accuracy_te = accuracy_score(y_true_test, y_pred_test)
    f1_te = f1_score(y_true_test, y_pred_test)
    recall_te = recall_score(y_true_test, y_pred_test)
    precision_te = precision_score(y_true_test, y_pred_test)
    roc_auc_te = roc_auc_score(y_true_test, y_prob_test)
    print(f"Test after training - Accuracy: {accuracy_te:.3f}, F1: {f1_te:.3f}, Recall: {recall_te:.3f}, Precision: {precision_te:.3f}, ROC-AUC: {roc_auc_te:.3f}")



In [6]:
if __name__ == "__main__":
    data_root = data_root_dir
    train_on_single_study(
        abundance_file="mpa4_species_profile_preprocessed.csv",
        metadata_file="sample_group_species_preprocessed.csv",
        study_name="QinJ_2012",        # single study to train/test
        data_root_dir=data_root,
        n_epochs=100,
        batch_size=16,
        lr=1e-3,
        model_script="src.models.models",
        model_name="model1",
        test_fraction=0.2,
        scale_factor_before_training=1000.0,
    )


Test before training - Accuracy: 0.541, F1: 0.227, Recall: 0.147, Precision: 0.500, ROC-AUC: 0.569
[Epoch 1/100] Training loss: 0.7405
[Epoch 2/100] Training loss: 0.6928
[Epoch 3/100] Training loss: 0.6626
[Epoch 4/100] Training loss: 0.6069
[Epoch 5/100] Training loss: 0.5700
[Epoch 6/100] Training loss: 0.5402
[Epoch 7/100] Training loss: 0.4670
[Epoch 8/100] Training loss: 0.6227
[Epoch 9/100] Training loss: 0.4376
[Epoch 10/100] Training loss: 0.4252
[Epoch 11/100] Training loss: 0.3987
[Epoch 12/100] Training loss: 0.3987
[Epoch 13/100] Training loss: 0.3567
[Epoch 14/100] Training loss: 0.3361
[Epoch 15/100] Training loss: 0.3143
[Epoch 16/100] Training loss: 0.2703
[Epoch 17/100] Training loss: 0.2192
[Epoch 18/100] Training loss: 0.2762
[Epoch 19/100] Training loss: 0.2195
[Epoch 20/100] Training loss: 0.1460
[Epoch 21/100] Training loss: 0.1388
[Epoch 22/100] Training loss: 0.2295
[Epoch 23/100] Training loss: 0.2677
[Epoch 24/100] Training loss: 0.1366
[Epoch 25/100] Trainin

In [7]:
if __name__ == "__main__":
    data_root = data_root_dir
    train_on_single_study(
        abundance_file="mpa4_species_profile_preprocessed.csv",
        metadata_file="sample_group_species_preprocessed.csv",
        study_name="QinJ_2012",        # single study to train/test
        data_root_dir=data_root,
        n_epochs=10,
        batch_size=16,
        lr=1e-3,
        model_script="src.models.models",
        model_name="model1",
        test_fraction=0.2,
        scale_factor_before_training=1000.0,
    )


Test before training - Accuracy: 0.568, F1: 0.556, Recall: 0.588, Precision: 0.526, ROC-AUC: 0.588
[Epoch 1/10] Training loss: 0.6817
[Epoch 2/10] Training loss: 0.6738
[Epoch 3/10] Training loss: 0.6622
[Epoch 4/10] Training loss: 0.6027
[Epoch 5/10] Training loss: 0.5219
[Epoch 6/10] Training loss: 0.5010
[Epoch 7/10] Training loss: 0.4763
[Epoch 8/10] Training loss: 0.4915
[Epoch 9/10] Training loss: 0.4380
[Epoch 10/10] Training loss: 0.4082
Train - Accuracy: 0.746, F1: 0.774, Recall: 0.841, Precision: 0.716, ROC-AUC: 0.841
Test after training - Accuracy: 0.635, F1: 0.400, Recall: 0.265, Precision: 0.818, ROC-AUC: 0.753
